# Benchmark institutionnel des formulations de Markowitz

**Objectif.** Isoler l'impact des **contraintes**, **régularisations** et **paramètres de
calibration** sur la robustesse et la **faisabilité opérationnelle** des portefeuilles, de la
formulation mathématique (QP/SOCP) au comportement **hors échantillon** par régime.

**5 formulations comparées (Phase 2).**
1. **Standard** : $\max_w\ \mu^\top w-\tfrac{\gamma}{2}w^\top\Sigma w$ s.c. $1^\top w=1$.
2. **Contraintes linéaires** : + bornes $l\le w\le u$ et levier $\sum|w_i|\le L$.
3. **Opérationnel** : + turnover $\|w-w_0\|_1\le T$ et coûts $\kappa^\top|w-w_0|$ dans l'objectif.
4. **Robust** : + pénalité d'incertitude sur $\mu$ : $-\kappa_\mu\|\Omega^{1/2}w\|_2$ (SOCP).
5. **Markowitz++** : tout simultanément, **calibré** 

## 1. Données et configuration

In [ ]:
import sys, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, ".."); sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import portfolio_lab as pl
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

USE_SYNTHETIC = False
START, END = "2007-01-01", "2025-12-31"

In [ ]:
constituents = pl.get_sp500_constituents()
#SELECTOR = pl.StratifiedBySector(n_total=80, seed=42)
SELECTOR = pl.AllUniverse()
# SELECTOR = pl.TopN(50)             # N premiers (ALPHABÉTIQUE)
# SELECTOR = pl.RandomN(50, seed=42) # aléatoire reproductible
universe = SELECTOR.select(constituents)
if USE_SYNTHETIC:
    prices = pl.make_synthetic_prices(constituents.loc[universe], START, END)
else:
    prices = pl.clean_prices(pl.download_prices(universe, START, END))
print(f"Univers : {prices.shape[1]} actifs | {prices.shape[0]} obs "
      f"({prices.index[0].date()} -> {prices.index[-1].date()})")

# Estimateurs FIGÉS (on isole l'effet de la FORMULATION, pas de μ/Σ).
MEAN_EST = pl.SampleMean()
COV_EST  = pl.EmpiricalCovariance()
WINDOW, STEP, TC_BPS, RF = 252, 21, 10.0, 0.04

## 2. Benchmark des 5 formulations (hors échantillon)


In [ ]:
variants = {
    "1. Standard":     pl.make_variant("standard"),
    "2. Linéaire":     pl.make_variant("linear", w_max=0.10),
    "3. Opérationnel": pl.make_variant("operational", w_max=0.10, tc_bps=TC_BPS),
    "4. Robust":       pl.make_variant("robust", w_max=0.10),
    "5. Markowitz++":  pl.make_variant("pp", w_max=0.10, tc_bps=TC_BPS),
}
t0 = time.time()
res = pl.run_markowitz_benchmark(prices, variants, mean_estimator=MEAN_EST, cov_estimator=COV_EST,
                                 window=WINDOW, step=STEP, tc_bps=TC_BPS, rf_annual=RF)
print(f"5 backtests en {time.time()-t0:.1f}s")
tbl = pl.markowitz_metrics_table(res)
display(tbl.round(3))

In [ ]:
# Courbes de valeur hors échantillon (base 1).
fig, ax = plt.subplots(figsize=(13, 6))
for name, r in res.items():
    ax.plot(r.history.index, r.history["nav"], lw=1.8, label=name)
ax.set_title("Formulations de Markowitz — valeur hors échantillon (base 1)")
ax.set_ylabel("NAV"); ax.legend(); plt.show()

In [ ]:
# Réalisme opérationnel : turnover moyen vs coûts cumulés vs stabilité.
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
tbl["Average turnover"].plot(kind="bar", ax=axes[0], color="indianred", edgecolor="black")
axes[0].set_title("Turnover moyen (↓ mieux)")
tbl["Cumulative cost (%)"].plot(kind="bar", ax=axes[1], color="slateblue", edgecolor="black")
axes[1].set_title("Coûts de transaction cumulés % (↓)")
tbl["Sharpe"].plot(kind="bar", ax=axes[2], color="seagreen", edgecolor="black")
axes[2].set_title("Sharpe net (↑)")
for a in axes: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 3. Analyse par **régime de marché**
On découpe la performance OOS par fenêtres datées : **krach COVID** (fév–avr 2020),
**resserrement 2022**, **marché calme 2023-2024**. Un bon Markowitz institutionnel résiste en
crise (drawdown contenu).

In [ ]:
REGIMES = {
    "COVID (fév-avr 2020)": ("2020-02-15", "2020-04-30"),
    "Resserrement 2022":    ("2022-01-01", "2022-12-31"),
    "Calme 2023-2024":      ("2023-01-01", "2024-12-31"),
}
def regime_metrics(ret, lo, hi):
    r = ret.loc[lo:hi].dropna()
    if len(r) < 5: return {"Sharpe": np.nan, "Vol %": np.nan, "MaxDD %": np.nan}
    nav = (1+r).cumprod()
    vol = r.std(ddof=1)*np.sqrt(252)
    cagr = nav.iloc[-1]**(252/len(r)) - 1
    mdd = (nav/nav.cummax()-1).min()
    return {"Sharpe": (cagr-RF)/vol if vol>0 else np.nan, "Vol %": vol*100, "MaxDD %": mdd*100}

for reg,(lo,hi) in REGIMES.items():
    print(f"\n=== {reg} ===")
    rows = {n: regime_metrics(r.history["strategy_return"], lo, hi) for n,r in res.items()}
    display(pd.DataFrame(rows).T.round(3))

## 4. Tuned Markowitz++

Calibration par **montée de coordonnées** (+25 % / −20 %) des hyperparamètres
$(\gamma_{\text{risk}},\gamma_{\text{trade}})$ sur la période **In-Sample** (≤ 2022), puis
application **Out-of-Sample** (2023-2025). On compare le ++ **calibré** au ++ par défaut.

In [ ]:
IS_END = "2022-12-31"
prices_is = prices.loc[:IS_END]
prices_oos = prices.loc["2022-01-01":]      # 1 an de recouvrement pour amorcer la fenêtre

t0 = time.time()
pen_star, score_star, hist = pl.calibrate_markowitzpp(
    prices_is, pl.make_variant("pp", w_max=0.10, tc_bps=TC_BPS),
    mean_estimator=MEAN_EST, cov_estimator=COV_EST,
    params=("gamma_risk", "gamma_trade"), window=WINDOW, step=STEP,
    tc_bps=TC_BPS, rf_annual=RF, max_passes=3)
print(f"Calibration IS en {time.time()-t0:.1f}s | score={score_star:.3f} | évaluations={len(hist)}")
print(f"γ* : gamma_risk={pen_star.gamma_risk:.3f}, gamma_trade={pen_star.gamma_trade:.3f}")

In [ ]:
# OOS : ++ calibré vs ++ défaut vs Opérationnel
tuned = pl.MarkowitzPlusPlus(pl.make_variant("pp", w_max=0.10).constraints, pen_star, TC_BPS)
oos = pl.run_markowitz_benchmark(prices_oos, {
    "Markowitz++ (tuned)":   tuned,
    "Markowitz++ (défaut)":  pl.make_variant("pp", w_max=0.10, tc_bps=TC_BPS),
    "Opérationnel":          pl.make_variant("operational", w_max=0.10, tc_bps=TC_BPS),
}, mean_estimator=MEAN_EST, cov_estimator=COV_EST, window=WINDOW, step=STEP, tc_bps=TC_BPS, rf_annual=RF)
display(pl.markowitz_metrics_table(oos).round(3))
print("Si le ++ calibré bat le ++ défaut en Sharpe NET OOS, la calibration a apporté de la valeur (H3).")

## 5. Analyse de **sensibilité** (±20 % sur μ et Σ)
Variation L1 des poids optimaux sous perturbation ±20 % des inputs. Un portefeuille robuste est
**peu sensible** (mean_l1 faible) → moins d'overfitting aux erreurs d'estimation.

In [ ]:
win = pl.compute_returns(prices.iloc[-WINDOW:], "simple")
mu = MEAN_EST.estimate(win)*252
Sig = COV_EST.estimate(win).values*252
sens = {}
for name, opt in variants.items():
    s = pl.weight_sensitivity(mu, Sig, opt, pct=0.20, n_dirs=20)
    sens[name] = {"mean_l1": s["mean_l1"], "max_l1": s["max_l1"]}
sens = pd.DataFrame(sens).T
display(sens.round(4))

fig, ax = plt.subplots(figsize=(11, 5))
sens["mean_l1"].plot(kind="bar", ax=ax, color="darkorange", edgecolor="black")
ax.set_title("Sensibilité des poids à ±20% des inputs (↓ = plus robuste)")
ax.tick_params(axis="x", rotation=30); plt.show()

## Matrice de décision & recommandations

In [ ]:
# Coût computationnel : temps d'UNE résolution par variante.
solve_time = {}
for name, opt in variants.items():
    t0 = time.time()
    for _ in range(5): opt.solve(mu, Sig, w_prev=np.ones(len(mu))/len(mu), mu_uncertainty=np.diag(Sig))
    solve_time[name] = (time.time()-t0)/5*1000   # ms

# Scores qualitatifs (simplicité, intérêt institutionnel) — propres à la formulation.
qual = {
    "1. Standard":     {"Simplicité": 5, "Intérêt instit.": 2},
    "2. Linéaire":     {"Simplicité": 4, "Intérêt instit.": 3},
    "3. Opérationnel": {"Simplicité": 3, "Intérêt instit.": 5},
    "4. Robust":       {"Simplicité": 3, "Intérêt instit.": 4},
    "5. Markowitz++":  {"Simplicité": 1, "Intérêt instit.": 5},
}
decision = pd.DataFrame({
    "Perf nette OOS (Sharpe ↑)":  tbl["Sharpe"].rank(ascending=False),
    "Robustesse (sensib. ↓)":     sens["mean_l1"].rank(ascending=True),
    "Faisabilité (turnover ↓)":   tbl["Average turnover"].rank(ascending=True),
    "Coût calcul (ms ↓)":         pd.Series(solve_time).rank(ascending=True),
    "Simplicité":                 pd.Series({k:v["Simplicité"] for k,v in qual.items()}).rank(ascending=False),
    "Intérêt institutionnel":     pd.Series({k:v["Intérêt instit."] for k,v in qual.items()}).rank(ascending=False),
})
decision["RANG MOYEN"] = decision.mean(axis=1)
display(decision.sort_values("RANG MOYEN").round(2))
print("\nTemps de résolution (ms):", {k: round(v,1) for k,v in solve_time.items()})
print("Meilleur compromis (rang moyen) :", decision["RANG MOYEN"].idxmin())

# Journalisation du run (`results/`)



In [ ]:
from portfolio_lab.reporting import RunReport

rep = RunReport("markowitz", results_dir="results")
rep.capture(
    prices=prices, selector=SELECTOR,
    requested_n=getattr(SELECTOR, "n_total", getattr(SELECTOR, "n", None)),
    period=(START, END), use_synthetic=USE_SYNTHETIC, w_max=0.10,
    mean_estimators={"μ": MEAN_EST}, cov_estimators={"Σ": COV_EST},
    extra={
        "Fenêtre d'estimation": f"{WINDOW} périodes",
        "Pas / détention": f"{STEP} périodes",
        "Coûts de transaction": f"{TC_BPS} bps",
        "Taux sans risque (annuel)": f"{RF:.2%}",
        "Formulations comparées": "; ".join(variants),
    },
)
ns = globals()
rep.add_if(ns, "res",  "Backtest des formulations (OOS)")
rep.add_if(ns, "oos",  "Calibration IS → OOS (++ tuned vs défaut)")
rep.add_if(ns, "sens", "Sensibilité des poids (±20 % sur μ et Σ)")
rep.add_if(ns, "qual", "Scores qualitatifs (simplicité / intérêt instit.)")
print("Rapport écrit :", rep.save())   # rep.save(fmt="both") pour tenter un PDF (pandoc)
